# Fase 07 - MCP Git e PR (Processo de Implementacao)

Este notebook documenta o padrao de implementacao do MCP para Git (status/commit/push) e Pull Request (status/create) no monorepo.

## 1) Objetivo

- Expor operacoes de fluxo de versao via HTTP no app principal.
- Padronizar comandos para status, commit, push e PR.
- Garantir validacao de ambiente (git e gh) com mensagens claras.
- Manter processo reproduzivel para evolucoes futuras.

## 2) Artefatos Implementados

1. Controller HTTP do MCP Git:
   - apps/monorepo-ai-llm/src/git-mcp.controller.ts
2. Service de execucao de comandos Git/GH:
   - apps/monorepo-ai-llm/src/git-mcp.service.ts
3. Registro no modulo principal:
   - apps/monorepo-ai-llm/src/app.module.ts

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
required_files = [
    'apps/monorepo-ai-llm/src/git-mcp.controller.ts',
    'apps/monorepo-ai-llm/src/git-mcp.service.ts',
    'apps/monorepo-ai-llm/src/app.module.ts',
]

for rel in required_files:
    p = ROOT / rel
    print(('OK' if p.exists() else 'MISSING'), rel)

## 3) Endpoints Disponiveis

Base path: /mcp/git

- GET /mcp/git/status
- POST /mcp/git/commit
- POST /mcp/git/push
- GET /mcp/git/pr/status
- POST /mcp/git/pr/create

## 4) Pre-condicoes de Ambiente

- Node.js e npm instalados
- Dependencias do projeto instaladas (npm install)
- Git instalado e repo inicializado
- GitHub CLI (gh) instalado para operacoes de PR
- gh autenticado (gh auth login)

In [ ]:
import shutil
import subprocess

checks = [
    ('node', ['node', '-v']),
    ('npm', ['npm', '-v']),
    ('git', ['git', '--version']),
    ('gh', ['gh', '--version']),
]

for name, cmd in checks:
    print(f'\n== {name} ==')
    if shutil.which(cmd[0]) is None:
        print('MISSING')
        continue
    res = subprocess.run(cmd, text=True, capture_output=True)
    print('exit:', res.returncode)
    out = res.stdout.strip() or res.stderr.strip()
    print(out if out else '(sem saida)')

## 5) Passos de Implementacao (Padrao)

1. Criar service com execucao segura de comandos (spawnSync) e validacao de retorno.
2. Criar controller com rotas REST para orquestrar operacoes.
3. Registrar controller/service no AppModule.
4. Compilar (npm run build).
5. Subir app em porta livre e testar rotas com curl.
6. Registrar falhas de ambiente com erro orientativo (ex.: gh nao encontrado).

In [ ]:
import subprocess

commands = [
    ['npm', 'run', 'build'],
]

for cmd in commands:
    print('>',' '.join(cmd))
    res = subprocess.run(cmd, text=True, capture_output=True)
    print('exit:', res.returncode)
    if res.stdout.strip():
        print(res.stdout.strip())
    if res.stderr.strip():
        print('stderr:')
        print(res.stderr.strip())

## 6) Exemplos de Teste HTTP

Com a API rodando na porta 3011:

```bash
curl -sS http://localhost:3011/mcp/git/status
curl -sS http://localhost:3011/mcp/git/pr/status
curl -sS -X POST http://localhost:3011/mcp/git/commit \
  -H 'Content-Type: application/json' \
  -d '{"message":"chore: update docs","addAll":false}'
curl -sS -X POST http://localhost:3011/mcp/git/push \
  -H 'Content-Type: application/json' \
  -d '{"remote":"origin","branch":"main"}'
curl -sS -X POST http://localhost:3011/mcp/git/pr/create \
  -H 'Content-Type: application/json' \
  -d '{"title":"feat: mcp git","base":"main","head":"feature/mcp-git"}'
```

## 7) Instalacao do GH (Usuario Local, sem sudo)

Use este fluxo quando o servidor nao tiver permissao de root:

```bash
mkdir -p ~/.local/bin ~/.local/opt
cd ~/.local/opt
curl -fsSL -o gh.tar.gz https://github.com/cli/cli/releases/latest/download/gh_2.49.2_linux_amd64.tar.gz
tar -xzf gh.tar.gz
cp gh_2.49.2_linux_amd64/bin/gh ~/.local/bin/gh
chmod +x ~/.local/bin/gh
export PATH="$HOME/.local/bin:$PATH"
gh --version
gh auth login
```

## 8) Troubleshooting

- TS1272 em controller: usar `import type` para tipos usados em parametros decorados.
- `gh: command not found`: instalar gh e atualizar PATH.
- `gh auth status` falha: executar `gh auth login`.
- `EADDRINUSE`: subir app em outra porta (`PORT=3011 npm run start`).
- PR endpoint retorna erro: validar branch atual, remote e autenticacao GitHub.

## 9) Checklist de Conclusao

- [ ] Build sem erros
- [ ] Endpoint /mcp/git/status responde
- [ ] Endpoint /mcp/git/pr/status responde com gh autenticado
- [ ] Endpoint /mcp/git/pr/create cria PR de teste
- [ ] Processo documentado e versionado